In [36]:
import uproot
import polars as pl
import numpy as np
import awkward as ak
import matplotlib.pyplot as plt

In [2]:
root_file_path = "pixel_singles_a_1_j_0.root"
# 1. Scan for valid trees
with uproot.open(root_file_path) as root_file:
    valid_trees = [
        f"{root_file_path}:Pixel_{i}_Singles" 
        for i in range(1, 74) 
        if f"Pixel_{i}_Singles" in root_file
    ]

# 2. Read into Awkward Array using multithreading
# library="ak" is uproot's default and fastest backend
ak_array = uproot.concatenate(valid_trees, library="ak", num_workers=4)

# 3. Convert to Polars via PyArrow (zero-copy transfer)
arrow_table = ak.to_arrow_table(ak_array)
df = pl.from_arrow(arrow_table)

print(f"Loaded Polars DataFrame with shape: {df.shape}")

Loaded Polars DataFrame with shape: (69241481, 15)


/tmp/ipykernel_635370/814625872.py:16: UserWarning: Extension type 'awkward' is not registered; loading as its storage type.

To avoid this warning, register the extension type or set environment variable 'POLARS_UNKNOWN_EXTENSION_TYPE_BEHAVIOR' to 'load_as_storage' or 'load_as_extension'.

In Polars 2.0, the default behavior will change to 'load_as_extension'.
  df = pl.from_arrow(arrow_table)


In [42]:
import json
# Read and parse the stats file, which is in JSON format
stats_file_path = "a_1_j_0_sim_stats.txt"
with open(stats_file_path, 'r') as f:
    stats = json.load(f)
# Flatten to extract just the values, ignoring the 'unit' keys for now
# This creates a clean dictionary: {'runs': 100, 'events': 500019668, ...}
flat_stats = {key: content["value"] for key, content in stats.items()}

print(f"Total events processed: {flat_stats['events']}")
print(f"Simulation duration: {flat_stats['duration']} hours")
print(f"Sensitivity: {df.shape[0]/flat_stats['events']*100:.6f}%")

Total events processed: 50000109080
Simulation duration: 12.0777 hours
Sensitivity: 0.138483%


In [41]:
# simulation throughoutput rate
total_events = flat_stats['events']
simulation_duration_hours = flat_stats['duration']
throughput_rate = total_events / (simulation_duration_hours*3600) # events per second
print(f"Simulation throughput rate: {throughput_rate:,.2f} events/second")
print(f"Stats file pps:             {flat_stats['pps']:,.2f}")

Simulation throughput rate: 1,149,963.92 events/second
Stats file pps:             1,149,960.00


In [33]:
# Read from the csv file the geometrical definition of the crystal and crystal
def _load_brain_spect_geometry():
    csv_path = Path(
        "../../../persistent_data/brain_spect/csv/BrainSPECT_Point_Cloud.007.25mmx0.556mm_pinhole.csv"
    )
    geometry_df = pl.read_csv(csv_path)
    geometry_df = geometry_df.with_columns(
        Pinhole_y=pl.col("Pinhole_z"),
        Pinhole_z=pl.col("Pinhole_y"),
        Crystal_y=pl.col("Crystal_z"),
        Crystal_z=pl.col("Crystal_y"),
    )
    elevation = np.arctan2(
        geometry_df["Pinhole_z"],
        np.sqrt(geometry_df["Pinhole_x"] ** 2 + geometry_df["Pinhole_y"] ** 2),
    )
    azimuth = (
        np.arctan2(geometry_df["Pinhole_y"], geometry_df["Pinhole_x"]) + 2 * np.pi
    ) % (2 * np.pi)
    geometry_df = geometry_df.with_columns(
        pl.Series("elevation", elevation),
        pl.Series("azimuth", azimuth),
    )
    geometry_df = geometry_df.with_columns(
        pl.col("elevation").round(6),
        pl.col("azimuth").round(6),
    ).sort(["elevation", "azimuth"])
    # Add a column for Pinhole to center distance
    geometry_df = geometry_df.with_columns(
        np.sqrt(
            geometry_df["Pinhole_x"] ** 2
            + geometry_df["Pinhole_y"] ** 2
            + geometry_df["Pinhole_z"] ** 2
        ).alias("Pinhole_distance")
    )
    return geometry_df
brain_spect_geometry_df = _load_brain_spect_geometry()
print(brain_spect_geometry_df.head())

shape: (5, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬──────────┬───────────┐
│ Pinhole_x ┆ Pinhole_y ┆ Pinhole_z ┆ Crystal_x ┆ … ┆ Crystal_z ┆ elevation ┆ azimuth  ┆ Pinhole_d │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---      ┆ istance   │
│ f64       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64      ┆ ---       │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆          ┆ f64       │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪══════════╪═══════════╡
│ 118.19505 ┆ 17.502012 ┆ -90.03754 ┆ 141.89569 ┆ … ┆ -108.0919 ┆ -0.645772 ┆ 0.147009 ┆ 149.61    │
│ 9         ┆           ┆ 6         ┆ 3         ┆   ┆ 96        ┆           ┆          ┆           │
│ 96.05995  ┆ 71.05546  ┆ -90.03754 ┆ 115.32202 ┆ … ┆ -108.0919 ┆ -0.645772 ┆ 0.636876 ┆ 149.61    │
│           ┆           ┆ 6         ┆ 2         ┆   ┆ 96        ┆           ┆

In [35]:
sens_est_geom = 73*2.007**2/(4*np.pi*149.61**2)
sens_sim = df.shape[0]/flat_stats['events']
print(f"Estimated sensitivity from geometry: {sens_est_geom*100:.6f}%")
print(f"Simulated sensitivity: {sens_sim*100:.6f}%")


Estimated sensitivity from geometry: 0.104541%
Simulated sensitivity: 0.138483%


In [6]:
# Print the column names
print("Columns in the DataFrame:")
print(df.columns)

Columns in the DataFrame:
['RunID', 'EventID', 'TotalEnergyDeposit', 'PostPosition_X', 'PostPosition_Y', 'PostPosition_Z', 'PrePosition_X', 'PrePosition_Y', 'PrePosition_Z', 'EventPosition_X', 'EventPosition_Y', 'EventPosition_Z', 'GlobalTime', 'PreStepUniqueVolumeID', 'PreStepUniqueVolumeIDAsInt']


In [7]:
# Print the TotalEnergyDeposit range
tot_energy_min = df["TotalEnergyDeposit"].min()
tot_energy_max = df["TotalEnergyDeposit"].max()
print(f"TotalEnergyDeposit range: {tot_energy_min} to {tot_energy_max}")

TotalEnergyDeposit range: 1.0118925863888961e-08 to 0.14000000000000004


In [8]:
# Print the Max and Min of EventPosition_X, EventPosition_Y, and EventPosition_Z
event_pos_x_min = df["EventPosition_X"].min()
event_pos_x_max = df["EventPosition_X"].max()
event_pos_y_min = df["EventPosition_Y"].min()
event_pos_y_max = df["EventPosition_Y"].max()
event_pos_z_min = df["EventPosition_Z"].min()
event_pos_z_max = df["EventPosition_Z"].max()
print(f"EventPosition_X range: {event_pos_x_min} to {event_pos_x_max}")
print(f"EventPosition_Y range: {event_pos_y_min} to {event_pos_y_max}")
print(f"EventPosition_Z range: {event_pos_z_min} to {event_pos_z_max}")

EventPosition_X range: -79.99999796814976 to 79.99999961591692
EventPosition_Y range: -79.99999857488272 to 79.99999577053148
EventPosition_Z range: -79.9999998480727 to 79.99999889149285


In [9]:
# Parse the PreStepUniqueVolumeID, it has the format:
# pixel_1_param-0_0_263, we want the first number and the last number
# first number is the Crystal ID, last number is the Pixel ID
df = df.with_columns(
  df["PreStepUniqueVolumeID"].str.split("_").list.get(1).cast(pl.Int32).alias("CrystalID"),
  df["PreStepUniqueVolumeID"].str.split("_").list.get(-1).cast(pl.Int32).alias("PixelID"),
)
# Move the new columns to the front for better visibility
df = df.select(["CrystalID", "PixelID"] + [col for col in df.columns if col not in ["CrystalID", "PixelID"]])
print(df.head())
print("Unique CrystalID values:", df["CrystalID"].unique().to_list())

shape: (5, 17)
┌───────────┬─────────┬───────┬──────────┬───┬─────────────┬────────────┬─────────────┬────────────┐
│ CrystalID ┆ PixelID ┆ RunID ┆ EventID  ┆ … ┆ EventPositi ┆ GlobalTime ┆ PreStepUniq ┆ PreStepUni │
│ ---       ┆ ---     ┆ ---   ┆ ---      ┆   ┆ on_Z        ┆ ---        ┆ ueVolumeID  ┆ queVolumeI │
│ i32       ┆ i32     ┆ i32   ┆ i32      ┆   ┆ ---         ┆ f64        ┆ ---         ┆ DAsInt     │
│           ┆         ┆       ┆          ┆   ┆ f64         ┆            ┆ str         ┆ ---        │
│           ┆         ┆       ┆          ┆   ┆             ┆            ┆             ┆ i32        │
╞═══════════╪═════════╪═══════╪══════════╪═══╪═════════════╪════════════╪═════════════╪════════════╡
│ 1         ┆ 29      ┆ 0     ┆ 7340825  ┆ … ┆ -1.738192   ┆ 2.0273e7   ┆ pixel_1_par ┆ 1          │
│           ┆         ┆       ┆          ┆   ┆             ┆            ┆ am-0_0_29   ┆            │
│ 1         ┆ 408     ┆ 0     ┆ 11908957 ┆ … ┆ -2.734658   ┆ 3.3876e7   ┆ pi

In [10]:
# filter by TotalEnergyDeposit
primary_energy = 0.14  # MeV
df_filtered = df.filter(
    (df["TotalEnergyDeposit"] >= primary_energy*0.999) & (df["TotalEnergyDeposit"] <= primary_energy*1.001)
)


In [11]:
# Histogram EventPosition_X, EventPosition_Y with filter 
# on CrystalID on energy filtered dataframe
from pathlib import Path
output_dir = Path("figs")
output_dir.mkdir(parents=True, exist_ok=True)
for crystal_id in df_filtered["CrystalID"].unique().to_list():
    df_crystal = df_filtered.filter(df_filtered["CrystalID"] == crystal_id)
    hist2d, edges_x, edges_y = np.histogram2d(
        df_crystal["EventPosition_X"].to_numpy(),
        df_crystal["EventPosition_Y"].to_numpy(),
        bins=80,
        range=[[-80, 80], [-80, 80]]
    )
    fig, ax = plt.subplots(figsize=(8, 6))
    # Use pcolormesh to create the heatmap
    pcm = ax.pcolormesh(edges_x, edges_y, hist2d.T, cmap='viridis', shading='auto')
    plt.colorbar(pcm, ax=ax, label='Counts')
    ax.set_xlabel('EventPosition_X (mm)')
    ax.set_ylabel('EventPosition_Y (mm)')
    ax.set_title(f'2D Histogram of Event Positions for CrystalID {crystal_id}')
    fig.savefig(f'figs/EventPosition_2D_Histogram_CrystalID_{crystal_id}.png')

    # load in the wrl in ../../brain_spect_geometry.wrl and project
    # the 3D geometry to 2D for visualization, then overlay the histogram

    plt.close(fig)

In [35]:
import qmirt
wrl_file_path = "../../brain_spect_geometry.wrl"
fig = qmirt.plot.wrl.plot_wrl_file(wrl_file_path)
fig.show()

In [ ]:
# Histogram EventPosition_X, EventPosition_Y with filter 
# on CrystalID on energy filtered dataframe
import re
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


def _polygon_bounds(polygons):
    xs = []
    ys = []
    for poly in polygons:
        for x, y in poly:
            xs.append(x)
            ys.append(y)
    if not xs:
        return None
    return min(xs), max(xs), min(ys), max(ys)


# 1. Parse the WRL file once, grouping 2D polygons by their associated Crystal ID
wrl_path = Path("brain_spect_geometry.wrl")
polygons_by_crystal = {}
geometry_bounds = None

if wrl_path.exists():
    with open(wrl_path, "r") as f:
        wrl_content = f.read()

    # Split the WRL content into blocks by the SOLID header
    solid_blocks = re.split(r'#---------- SOLID:\s+', wrl_content)

    for block in solid_blocks[1:]:  # Skip the header chunk
        # The solid name is on the first line of the split block
        first_newline = block.find('\n')
        solid_name = block[:first_newline].strip()

        # Match only Collimator and DetectorCrystal to skip pixel_X_param
        match = re.match(r'(Collimator|DetectorCrystal)_(\d+):0', solid_name)
        if not match:
            continue

        crystal_id = int(match.group(2))

        if crystal_id not in polygons_by_crystal:
            polygons_by_crystal[crystal_id] = []

        # Extract coordinate points and indices blocks for this solid
        points_match = re.search(r'point\s*\[(.*?)\]', block, re.DOTALL)
        indices_match = re.search(r'coordIndex\s*\[(.*?)\]', block, re.DOTALL)

        if points_match and indices_match:
            pts = []
            for line in points_match.group(1).split(','):
                nums = line.strip().split()
                if len(nums) == 3:
                    pts.append([float(nums[0]), float(nums[1])])

            idx_list = [int(num) for num in re.findall(r'-?\d+', indices_match.group(1))]

            poly = []
            for idx in idx_list:
                if idx == -1:
                    if poly:
                        polygons_by_crystal[crystal_id].append([pts[i] for i in poly])
                    poly = []
                else:
                    poly.append(idx)

    all_polygons = [poly for crystal_polygons in polygons_by_crystal.values() for poly in crystal_polygons]
    geometry_bounds = _polygon_bounds(all_polygons)
else:
    print(f"Warning: WRL file not found at {wrl_path.resolve()}")


# 2. Iterate through filtered dataframe and plot
output_dir = Path("figs")
output_dir.mkdir(parents=True, exist_ok=True)

x_min = df_filtered["EventPosition_X"].min()
x_max = df_filtered["EventPosition_X"].max()
y_min = df_filtered["EventPosition_Y"].min()
y_max = df_filtered["EventPosition_Y"].max()

extent_candidates = [
    abs(x_min),
    abs(x_max),
    abs(y_min),
    abs(y_max),
]

if geometry_bounds is not None:
    geom_x_min, geom_x_max, geom_y_min, geom_y_max = geometry_bounds
    extent_candidates.extend([
        abs(geom_x_min),
        abs(geom_x_max),
        abs(geom_y_min),
        abs(geom_y_max),
    ])

plot_extent = max(extent_candidates) if extent_candidates else 80.0
if not np.isfinite(plot_extent) or plot_extent <= 0:
    plot_extent = 80.0
plot_extent = max(plot_extent * 1.15, 100.0)

fov_min = -80.0
fov_max = 80.0
plot_min = -plot_extent
plot_max = plot_extent
hist_range = [[plot_min, plot_max], [plot_min, plot_max]]

for crystal_id in df_filtered["CrystalID"].unique().to_list():
    df_crystal = df_filtered.filter(df_filtered["CrystalID"] == crystal_id)
    hist2d, edges_x, edges_y = np.histogram2d(
        df_crystal["EventPosition_X"].to_numpy(),
        df_crystal["EventPosition_Y"].to_numpy(),
        bins=80,
        range=hist_range,
    )

    fig, ax = plt.subplots(figsize=(8, 8))

    # Use pcolormesh to create the heatmap
    pcm = ax.pcolormesh(edges_x, edges_y, hist2d.T, cmap='viridis', shading='auto')
    plt.colorbar(pcm, ax=ax, label='Counts')
    ax.set_xlabel('EventPosition_X (mm)')
    ax.set_ylabel('EventPosition_Y (mm)')
    ax.set_title(f'2D Histogram of Event Positions for CrystalID {crystal_id}')
    ax.set_xlim(plot_min, plot_max)
    ax.set_ylim(plot_min, plot_max)
    ax.set_aspect('equal', adjustable='box')

    # 3. Overlay the 2D WRL geometry specific to this crystal_id
    polygons_to_draw = polygons_by_crystal.get(int(crystal_id), [])

    for poly in polygons_to_draw:
        # Extract X and Y arrays, wrapping back to the first point to close the polygon
        xs = [p[0] for p in poly] + [poly[0][0]]
        ys = [p[1] for p in poly] + [poly[0][1]]
        ax.plot(xs, ys, color='red', linewidth=0.5, alpha=0.6)

    fig.savefig(output_dir / f'EventPosition_2D_Histogram_CrystalID_{crystal_id}.png', bbox_inches='tight')
    plt.close(fig)


In [30]:
# Crystal 1 per-pixel histograms, using the monolithic crystal outline only
import polars as pl
from pathlib import Path
from scipy.spatial.transform import Rotation


def _polygon_bounds(polygons):
    xs = []
    ys = []
    for poly in polygons:
        for x, y in poly:
            xs.append(x)
            ys.append(y)
    if not xs:
        return None
    return min(xs), max(xs), min(ys), max(ys)


def _load_brain_spect_geometry():
    csv_path = Path(
        "../../../persistent_data/brain_spect/csv/BrainSPECT_Point_Cloud.007.25mmx0.556mm_pinhole.csv"
    )
    geometry_df = pl.read_csv(csv_path)
    geometry_df = geometry_df.with_columns(
        Pinhole_y=pl.col("Pinhole_z"),
        Pinhole_z=pl.col("Pinhole_y"),
        Crystal_y=pl.col("Crystal_z"),
        Crystal_z=pl.col("Crystal_y"),
    )
    elevation = np.arctan2(
        geometry_df["Pinhole_z"],
        np.sqrt(geometry_df["Pinhole_x"] ** 2 + geometry_df["Pinhole_y"] ** 2),
    )
    azimuth = (
        np.arctan2(geometry_df["Pinhole_y"], geometry_df["Pinhole_x"]) + 2 * np.pi
    ) % (2 * np.pi)
    geometry_df = geometry_df.with_columns(
        pl.Series("elevation", elevation),
        pl.Series("azimuth", azimuth),
    )
    geometry_df = geometry_df.with_columns(
        pl.col("elevation").round(6),
        pl.col("azimuth").round(6),
    ).sort(["elevation", "azimuth"])
    return geometry_df


def _ray_box_segment_2d(origin_xy, direction_xy, lower, upper):
    epsilon = 1e-12
    t_enter = -np.inf
    t_exit = np.inf

    for axis in range(2):
        origin_value = float(origin_xy[axis])
        direction_value = float(direction_xy[axis])

        if abs(direction_value) < epsilon:
            if origin_value < lower or origin_value > upper:
                return None
            continue

        t0 = (lower - origin_value) / direction_value
        t1 = (upper - origin_value) / direction_value
        t_axis_min = min(t0, t1)
        t_axis_max = max(t0, t1)
        t_enter = max(t_enter, t_axis_min)
        t_exit = min(t_exit, t_axis_max)

        if t_enter > t_exit:
            return None

    if t_exit < 0:
        return None

    t_start = max(t_enter, 0.0)
    start_xy = origin_xy + t_start * direction_xy
    end_xy = origin_xy + t_exit * direction_xy
    return start_xy, end_xy


geometry_df = _load_brain_spect_geometry()


# 1. Parse the WRL file once, grouping 2D polygons by their associated Crystal ID
wrl_path = Path("brain_spect_geometry.wrl")
polygons_by_crystal = {}
geometry_bounds = None

if wrl_path.exists():
    with open(wrl_path, "r") as f:
        wrl_content = f.read()

    # Split the WRL content into blocks by the SOLID header
    solid_blocks = re.split(r"#---------- SOLID:\s+", wrl_content)

    for block in solid_blocks[1:]:  # Skip the header chunk
        # The solid name is on the first line of the split block
        first_newline = block.find("\n")
        solid_name = block[:first_newline].strip()

        # Match only Collimator and DetectorCrystal to skip pixel_X_param
        match = re.match(r"(Collimator|DetectorCrystal)_(\d+):0", solid_name)
        if not match:
            continue

        crystal_id = int(match.group(2))

        if crystal_id not in polygons_by_crystal:
            polygons_by_crystal[crystal_id] = []

        # Extract coordinate points and indices blocks for this solid
        points_match = re.search(r"point\s*\[(.*?)\]", block, re.DOTALL)
        indices_match = re.search(r"coordIndex\s*\[(.*?)\]", block, re.DOTALL)

        if points_match and indices_match:
            pts = []
            for line in points_match.group(1).split(","):
                nums = line.strip().split()
                if len(nums) == 3:
                    pts.append([float(nums[0]), float(nums[1])])

            idx_list = [
                int(num) for num in re.findall(r"-?\d+", indices_match.group(1))
            ]

            poly = []
            for idx in idx_list:
                if idx == -1:
                    if poly:
                        polygons_by_crystal[crystal_id].append([pts[i] for i in poly])
                    poly = []
                else:
                    poly.append(idx)

    all_polygons = [
        poly
        for crystal_polygons in polygons_by_crystal.values()
        for poly in crystal_polygons
    ]
    geometry_bounds = _polygon_bounds(all_polygons)
else:
    print(f"Warning: WRL file not found at {wrl_path.resolve()}")


# 2. Iterate through filtered dataframe and plot
output_dir = Path("figs/per_pixel")
output_dir.mkdir(parents=True, exist_ok=True)

x_min = df_filtered["EventPosition_X"].min()
x_max = df_filtered["EventPosition_X"].max()
y_min = df_filtered["EventPosition_Y"].min()
y_max = df_filtered["EventPosition_Y"].max()

extent_candidates = [
    abs(x_min),
    abs(x_max),
    abs(y_min),
    abs(y_max),
]

if geometry_bounds is not None:
    geom_x_min, geom_x_max, geom_y_min, geom_y_max = geometry_bounds
    extent_candidates.extend(
        [
            abs(geom_x_min),
            abs(geom_x_max),
            abs(geom_y_min),
            abs(geom_y_max),
        ]
    )

plot_extent = max(extent_candidates) if extent_candidates else 80.0
if not np.isfinite(plot_extent) or plot_extent <= 0:
    plot_extent = 80.0

fov_min = -80.0
fov_max = 80.0
plot_min = fov_min
plot_max = fov_max
hist_range = [[plot_min, plot_max], [plot_min, plot_max]]

crystal_id_focus = 1
head_idx = crystal_id_focus - 1
pixel_count_xy = 25
pixel_pitch_mm = np.array([50.0 / pixel_count_xy, 50.0 / pixel_count_xy, 10.0])
crystal_center = np.array(
    [
        geometry_df.item(head_idx, "Crystal_x"),
        geometry_df.item(head_idx, "Crystal_y"),
        geometry_df.item(head_idx, "Crystal_z"),
    ],
    dtype=float,
)
pinhole_position = np.array(
    [
        geometry_df.item(head_idx, "Pinhole_x"),
        geometry_df.item(head_idx, "Pinhole_y"),
        geometry_df.item(head_idx, "Pinhole_z"),
    ],
    dtype=float,
)
azimuth = geometry_df.item(head_idx, "azimuth")
elevation = geometry_df.item(head_idx, "elevation")
r_base_x = Rotation.from_euler("x", -90, degrees=True)
r_base_z = Rotation.from_euler("z", 90, degrees=True)
r_dyn_z = Rotation.from_euler("z", azimuth, degrees=False)
r_dyn_x = Rotation.from_euler("x", -elevation, degrees=False)
head_rotation = (r_dyn_z * r_base_z * r_dyn_x * r_base_x).as_matrix()

crystal_polygons = polygons_by_crystal.get(crystal_id_focus, [])

for pixel_id in range(1, 625):
    df_pixel = df_filtered.filter(
        (df_filtered["CrystalID"] == crystal_id_focus)
        & (df_filtered["PixelID"] == pixel_id)
    )

    hist2d, edges_x, edges_y = np.histogram2d(
        df_pixel["EventPosition_X"].to_numpy(),
        df_pixel["EventPosition_Y"].to_numpy(),
        bins=80,
        range=hist_range,
    )

    pixel_index = pixel_id - 1
    pixel_col = pixel_index // pixel_count_xy
    pixel_row = pixel_index % pixel_count_xy

    local_pixel_center = np.array(
        [
            0.5 * 50.0 - (pixel_col + 0.5) * pixel_pitch_mm[0],
            -0.5 * 50.0 + (pixel_row + 0.5) * pixel_pitch_mm[1],
            0.0,
        ]
    )

    preposition_center = None
    if df_pixel.height > 0:
        pre_x = df_pixel["PrePosition_X"].mean()
        pre_y = df_pixel["PrePosition_Y"].mean()
        pre_z = df_pixel["PrePosition_Z"].mean()
        if all(
            value is not None and np.isfinite(value) for value in [pre_x, pre_y, pre_z]
        ):
            preposition_center = np.array([pre_x, pre_y, pre_z], dtype=float)

    if preposition_center is not None:
        global_pixel_center = preposition_center
    else:
        global_pixel_center = crystal_center + head_rotation @ local_pixel_center

  


    fig, ax = plt.subplots(figsize=(8, 8))
    pcm = ax.pcolormesh(edges_x, edges_y, hist2d.T, cmap="viridis", shading="auto")
    plt.colorbar(pcm, ax=ax, label="Counts")
    ax.set_xlabel("EventPosition_X (mm)")
    ax.set_ylabel("EventPosition_Y (mm)")
    ax.set_title(
        f"Crystal {crystal_id_focus} - Pixel {pixel_id} "
        f"(row {pixel_row + 1}, col {pixel_col + 1})"
    )
    plot_min = -150
    plot_max = 150
    ax.set_xlim(plot_min, plot_max)
    ax.set_ylim(plot_min, plot_max)
    ax.set_aspect("equal", adjustable="box")

    # Draw the crystal outline from the WRL geometry.
    for poly in crystal_polygons:
        xs = [p[0] for p in poly] + [poly[0][0]]
        ys = [p[1] for p in poly] + [poly[0][1]]
        ax.plot(xs, ys, color="red", linewidth=0.5, alpha=0.6)

    # Draw the FOV square explicitly so the valid imaging window is visible.
    fov_xs = [fov_min, fov_max, fov_max, fov_min, fov_min]
    fov_ys = [fov_min, fov_min, fov_max, fov_max, fov_min]
    ax.plot(
        fov_xs, fov_ys, color="cyan", linewidth=1.0, linestyle="--", alpha=0.9, zorder=4
    )

    # Mark the pinhole and the selected pixel center, then connect them.
    ax.scatter(
        [pinhole_position[0]],
        [pinhole_position[1]],
        color="green",
        edgecolor="black",
        s=36,
        zorder=5,
        label="Pinhole",
    )
    ax.scatter(
        [global_pixel_center[0]],
        [global_pixel_center[1]],
        color="orange",
        edgecolor="black",
        s=30,
        zorder=5,
        label="Pixel center",
    )

    ray_segment = _ray_box_segment_2d(
        global_pixel_center[:2],
        pinhole_position[:2] - global_pixel_center[:2],
        plot_min,
        plot_max,
    )
    if ray_segment is not None:
        ray_start, ray_end = ray_segment
        ax.plot(
            [ray_start[0], ray_end[0]],
            [ray_start[1], ray_end[1]],
            color="green",
            linewidth=0.8,
            alpha=0.85,
            zorder=4,
        )

    fig.savefig(
        output_dir / f"Crystal_{crystal_id_focus}_Pixel_{pixel_id:03d}.png",
        bbox_inches="tight",
    )
    plt.close(fig)
